# Nghiệm thu SCD2 ngày 2 bằng Spark SQL

Chạy sau khi Bronze và Silver đã hoàn thành cho `2026-01-01`. Toàn bộ logic kiểm tra nằm trong Spark SQL. Vì kernel của dự án là Python nên mỗi câu SQL được đặt trong `spark.sql("""...""")`; phần Python còn lại chỉ dùng để hiển thị kết quả và dừng notebook khi có `FAIL`.

In [1]:
try:
    spark
except NameError as exc:
    raise RuntimeError("Hãy chọn kernel 'PySpark (Lakehouse)' trước khi chạy") from exc

## 1. Các quy tắc lịch sử chung

Câu SQL đầu tiên chuẩn hóa bốn dimension về cùng bộ cột, sau đó kiểm tra trùng current row, trùng surrogate key, khoảng hiệu lực sai và version bị overlap.

In [2]:
invariant_checks = spark.sql("""
WITH versions AS (
    SELECT 'dim_branch' AS table_name, CAST(branch_code AS STRING) AS business_key,
           CAST(branch_sk AS STRING) AS surrogate_key, effective_from, effective_to, is_current
    FROM lakehouse.silver.dim_branch
    UNION ALL
    SELECT 'dim_product', CAST(product_code AS STRING), CAST(product_sk AS STRING),
           effective_from, effective_to, is_current
    FROM lakehouse.silver.dim_product
    UNION ALL
    SELECT 'dim_customer', CAST(customer_id AS STRING), CAST(customer_sk AS STRING),
           effective_from, effective_to, is_current
    FROM lakehouse.silver.dim_customer
    UNION ALL
    SELECT 'dim_account', CAST(account_id AS STRING), CAST(account_sk AS STRING),
           effective_from, effective_to, is_current
    FROM lakehouse.silver.dim_account
),
ordered_versions AS (
    SELECT *,
           LAG(effective_to) OVER (
               PARTITION BY table_name, business_key
               ORDER BY effective_from, surrogate_key
           ) AS previous_effective_to
    FROM versions
),
violation_rows AS (
    SELECT table_name, 'multiple_current_rows' AS check_name, business_key AS item
    FROM versions
    WHERE is_current = 1
    GROUP BY table_name, business_key
    HAVING COUNT(*) > 1

    UNION ALL
    SELECT table_name, 'duplicate_surrogate_key', surrogate_key
    FROM versions
    GROUP BY table_name, surrogate_key
    HAVING COUNT(*) > 1

    UNION ALL
    SELECT table_name, 'invalid_effective_range', COALESCE(business_key, '<NULL>')
    FROM versions
    WHERE business_key IS NULL
       OR surrogate_key IS NULL
       OR effective_from IS NULL
       OR effective_to IS NULL
       OR is_current IS NULL
       OR effective_from > effective_to
       OR (is_current = 1 AND effective_to <> DATE '9999-12-31')
       OR (is_current = 0 AND effective_to = DATE '9999-12-31')

    UNION ALL
    SELECT table_name, 'overlapping_versions', business_key
    FROM ordered_versions
    WHERE previous_effective_to IS NOT NULL
      AND effective_from <= previous_effective_to
),
check_types AS (
    SELECT EXPLODE(ARRAY(
        'multiple_current_rows',
        'duplicate_surrogate_key',
        'invalid_effective_range',
        'overlapping_versions'
    )) AS check_name
),
expected_checks AS (
    SELECT dimensions.table_name, check_types.check_name
    FROM (SELECT DISTINCT table_name FROM versions) dimensions
    CROSS JOIN check_types
)
SELECT expected.table_name,
       expected.check_name,
       COUNT(violations.item) AS violation_count,
       CASE WHEN COUNT(violations.item) = 0 THEN 'PASS' ELSE 'FAIL' END AS status
FROM expected_checks expected
LEFT JOIN violation_rows violations
  ON expected.table_name = violations.table_name
 AND expected.check_name = violations.check_name
GROUP BY expected.table_name, expected.check_name
ORDER BY expected.table_name, expected.check_name
""")

invariant_checks.show(100, truncate=False)
assert invariant_checks.where("status = 'FAIL'").count() == 0, "Generic SCD2 invariant failed"

+------------+-----------------------+---------------+------+
|table_name  |check_name             |violation_count|status|
+------------+-----------------------+---------------+------+
|dim_account |duplicate_surrogate_key|0              |PASS  |
|dim_account |invalid_effective_range|0              |PASS  |
|dim_account |multiple_current_rows  |0              |PASS  |
|dim_account |overlapping_versions   |0              |PASS  |
|dim_branch  |duplicate_surrogate_key|0              |PASS  |
|dim_branch  |invalid_effective_range|0              |PASS  |
|dim_branch  |multiple_current_rows  |0              |PASS  |
|dim_branch  |overlapping_versions   |0              |PASS  |
|dim_customer|duplicate_surrogate_key|0              |PASS  |
|dim_customer|invalid_effective_range|0              |PASS  |
|dim_customer|multiple_current_rows  |0              |PASS  |
|dim_customer|overlapping_versions   |0              |PASS  |
|dim_product |duplicate_surrogate_key|0              |PASS  |
|dim_pro

## 2. Các key kiểm soát phải có version cũ và mới

View tạm bên dưới gom bốn key demo về cùng schema. Cột `business_values` cho thấy chính xác giá trị nghiệp vụ đã đổi.

In [3]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW acceptance_controlled_history AS
SELECT 'dim_branch' AS table_name, branch_code AS business_key,
       CAST(branch_sk AS STRING) AS surrogate_key, effective_from, effective_to, is_current,
       CONCAT('manager=', manager_name, '; address=', address, '; status=', status) AS business_values
FROM lakehouse.silver.dim_branch
WHERE branch_code = 'BDG001'

UNION ALL
SELECT 'dim_product', product_code, CAST(product_sk AS STRING), effective_from, effective_to, is_current,
       CONCAT('name=', product_name, '; is_active=', CAST(is_active AS STRING))
FROM lakehouse.silver.dim_product
WHERE product_code = 'CASA002'

UNION ALL
SELECT 'dim_customer', CAST(customer_id AS STRING), CAST(customer_sk AS STRING),
       effective_from, effective_to, is_current,
       CONCAT('phone=', phone, '; address=', address, '; segment=', customer_segment)
FROM lakehouse.silver.dim_customer
WHERE customer_id = 10001

UNION ALL
SELECT 'dim_account', CAST(account_id AS STRING), CAST(account_sk AS STRING),
       effective_from, effective_to, is_current,
       CONCAT('branch=', branch_code, '; status=', status, '; balance=', CAST(balance AS STRING))
FROM lakehouse.silver.dim_account
WHERE account_id = 100001
""")

spark.sql("""
SELECT *
FROM acceptance_controlled_history
ORDER BY table_name, effective_from
""").show(100, truncate=False)

controlled_checks = spark.sql("""
SELECT table_name,
       COUNT(*) AS versions,
       COUNT(DISTINCT surrogate_key) AS distinct_surrogate_keys,
       SUM(CASE WHEN is_current = 1 THEN 1 ELSE 0 END) AS current_versions,
       MIN(effective_from) AS first_effective_from,
       MAX(effective_from) AS last_effective_from,
       CASE
           WHEN COUNT(*) = 2
            AND COUNT(DISTINCT surrogate_key) = 2
            AND SUM(CASE WHEN is_current = 1 THEN 1 ELSE 0 END) = 1
            AND SUM(CASE WHEN is_current = 0 THEN 1 ELSE 0 END) = 1
            AND MIN(effective_from) = DATE '2025-12-31'
            AND MAX(effective_from) = DATE '2026-01-01'
            AND MAX(CASE WHEN is_current = 0 THEN effective_to END) = DATE '2025-12-31'
            AND MAX(CASE WHEN is_current = 1 THEN effective_to END) = DATE '9999-12-31'
           THEN 'PASS' ELSE 'FAIL'
       END AS status
FROM acceptance_controlled_history
GROUP BY table_name
ORDER BY table_name
""")

controlled_checks.show(100, truncate=False)
assert controlled_checks.where("status = 'FAIL'").count() == 0, "Controlled history check failed"

+------------+------------+----------------------------------------------------------------+--------------+------------+----------+--------------------------------------------------------------------------+
|table_name  |business_key|surrogate_key                                                   |effective_from|effective_to|is_current|business_values                                                           |
+------------+------------+----------------------------------------------------------------+--------------+------------+----------+--------------------------------------------------------------------------+
|dim_account |100001      |23cf690a9650fb0f63a7bd2033c9319ffadebd5ad2a87f928ac244ddb1a30eb0|2025-12-31    |2025-12-31  |0         |branch=BDG001; status=ACTIVE; balance=848602000.00                        |
|dim_account |100001      |a6aa99890e1b7eed0b35057528cddbdacdd5edc1c441862818bb4d536a8ea3ae|2026-01-01    |9999-12-31  |1         |branch=BDG002; status=FROZEN; balance=849

+------------+--------+-----------------------+----------------+--------------------+-------------------+------+
|table_name  |versions|distinct_surrogate_keys|current_versions|first_effective_from|last_effective_from|status|
+------------+--------+-----------------------+----------------+--------------------+-------------------+------+
|dim_account |2       |2                      |1               |2025-12-31          |2026-01-01         |PASS  |
|dim_branch  |2       |2                      |1               |2025-12-31          |2026-01-01         |PASS  |
|dim_customer|2       |2                      |1               |2025-12-31          |2026-01-01         |PASS  |
|dim_product |2       |2                      |1               |2025-12-31          |2026-01-01         |PASS  |
+------------+--------+-----------------------+----------------+--------------------+-------------------+------+



## 3. Thay đổi kỹ thuật không được tạo lịch sử

Customer `10002` chỉ đổi `last_updated` trong source.

In [4]:
technical_only_check = spark.sql("""
SELECT customer_id,
       COUNT(*) AS versions,
       SUM(CASE WHEN is_current = 1 THEN 1 ELSE 0 END) AS current_versions,
       CASE
           WHEN COUNT(*) = 1
            AND SUM(CASE WHEN is_current = 1 THEN 1 ELSE 0 END) = 1
           THEN 'PASS' ELSE 'FAIL'
       END AS status
FROM lakehouse.silver.dim_customer
WHERE customer_id = 10002
GROUP BY customer_id
""")

technical_only_check.show(truncate=False)
assert technical_only_check.where("status = 'FAIL'").count() == 0, "Technical-only change created history"

+-----------+--------+----------------+------+
|customer_id|versions|current_versions|status|
+-----------+--------+----------------+------+
|10002      |1       |1               |PASS  |
+-----------+--------+----------------+------+



## 4. Fingerprint kiểm tra idempotency

Cell đầu tạo fingerprint bằng SQL và lưu kết quả trước rerun. Sau đó rerun `silver_all_dag` với `cob_dt=2026-01-01`, chờ DAG thành công rồi chạy cell cuối để so sánh.

In [5]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW acceptance_dimension_fingerprint AS
WITH surrogate_keys AS (
    SELECT 'dim_branch' AS table_name, CAST(branch_sk AS STRING) AS surrogate_key
    FROM lakehouse.silver.dim_branch
    UNION ALL
    SELECT 'dim_product', CAST(product_sk AS STRING)
    FROM lakehouse.silver.dim_product
    UNION ALL
    SELECT 'dim_customer', CAST(customer_sk AS STRING)
    FROM lakehouse.silver.dim_customer
    UNION ALL
    SELECT 'dim_account', CAST(account_sk AS STRING)
    FROM lakehouse.silver.dim_account
)
SELECT table_name,
       COUNT(*) AS row_count,
       SHA2(CONCAT_WS('|', SORT_ARRAY(COLLECT_LIST(surrogate_key))), 256) AS surrogate_key_sha256
FROM surrogate_keys
GROUP BY table_name
""")

spark.sql("SELECT * FROM acceptance_dimension_fingerprint ORDER BY table_name").show(truncate=False)
before_rerun = spark.sql("SELECT * FROM acceptance_dimension_fingerprint ORDER BY table_name").collect()

+------------+---------+----------------------------------------------------------------+
|table_name  |row_count|surrogate_key_sha256                                            |
+------------+---------+----------------------------------------------------------------+
|dim_account |13501    |68fabd37f4f34ff3ce75dff4119c28bbf62de74ed37d817c314862431e708082|
|dim_branch  |51       |fb7a56bb88fa563626daa5d2db03fd4bde2865ac9fadf30aaa6e8b038e7e25d3|
|dim_customer|10001    |afc5be229f21d939d7818ee9f507c43532fef9665b652cf962dc58cef8369c62|
|dim_product |17       |677c1f89adf1becdb87d35d37d4b5f5a6fd5a3799eb148c9d486cf672ecdf66a|
+------------+---------+----------------------------------------------------------------+



In [6]:
spark.sql("SELECT * FROM acceptance_dimension_fingerprint ORDER BY table_name").show(truncate=False)
after_rerun = spark.sql("SELECT * FROM acceptance_dimension_fingerprint ORDER BY table_name").collect()
assert after_rerun == before_rerun, "Rerun changed row counts or surrogate-key sets"
print("IDEMPOTENCY: PASS")

+------------+---------+----------------------------------------------------------------+
|table_name  |row_count|surrogate_key_sha256                                            |
+------------+---------+----------------------------------------------------------------+
|dim_account |13501    |68fabd37f4f34ff3ce75dff4119c28bbf62de74ed37d817c314862431e708082|
|dim_branch  |51       |fb7a56bb88fa563626daa5d2db03fd4bde2865ac9fadf30aaa6e8b038e7e25d3|
|dim_customer|10001    |afc5be229f21d939d7818ee9f507c43532fef9665b652cf962dc58cef8369c62|
|dim_product |17       |677c1f89adf1becdb87d35d37d4b5f5a6fd5a3799eb148c9d486cf672ecdf66a|
+------------+---------+----------------------------------------------------------------+



IDEMPOTENCY: PASS
